# 11 — Columnas anidadas sin explotar (`attributes`, `pictures`, `variations`)

Hoy de estas tres columnas solo usamos **cuántos** elementos tienen (`n_atributos`, `n_fotos`, `n_variaciones`, en `aplanar_anidadas`). Acá miro **qué hay adentro** con un EDA rápido, y armo bloques de features **solo** para lo que el EDA justifique.

Motivación (`08_errores.ipynb`): más de la mitad de los errores son de vendedores con 1 publicación, donde el modelo depende del contenido de la publicación. Las fotos están en el 99% de las publicaciones, así que cualquier señal que tengan llega también a esos vendedores.

Crear variables nuevas que resuman información de otras es *feature engineering*, visto en `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb` (sección "Feature Engineering"). La regla de ese material: "es importante al generar una nueva variable saber con qué fin se genera y qué interpretación tiene". Por eso cada feature de abajo sale de algo que se ve en el EDA.

**Base de este paso = T2a de `09_texto.ipynb`** (Base anterior + `prob_texto_new` con `PrediccionOOF`) = 0,9291. Modelo fijo y CV iguales a `07`–`09`.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder, PrediccionOOF

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

resultados_bloques = []


def evaluar(nombre, pipeline):
    """Mide cualquier pipeline con la CV fija y guarda accuracy/roc_auc en resultados_bloques."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))


def armar_modelo_texto(C):
    vectores = ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
    ])
    return Pipeline([
        ("vectores", vectores),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


def armar_pipeline_hgb(columnas_numericas_extra=()):
    """Pipeline T2a (Base + prob_texto_new) con la opción de sumar columnas numéricas."""
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas_base + list(columnas_numericas_extra)),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto(C=2), random_state=SEED), ["title"]),
    ]
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", ColumnTransformer(transformers, sparse_threshold=0)),
        ("modelo", modelo_fijo),
    ])


def pct_new_por_grupo(grupos):
    """Cantidad de filas y % de new real por cada valor de `grupos` (Serie alineada con train)."""
    return pd.DataFrame({
        "n": grupos.value_counts(sort=False),
        "pct_del_total": grupos.value_counts(sort=False, normalize=True),
        "pct_new": y.groupby(grupos, observed=True).mean(),
    }).round(3)

## Base (T2a)

Tiene que dar 0,9291. Si no, paro.

In [2]:
evaluar("Base (T2a)", armar_pipeline_hgb())

Base (T2a) -> accuracy: 0.9291 +/- 0.0017 | roc_auc: 0.9807


## EDA 1 — `attributes`

Cada fila es una lista de diccionarios `{"id", "name", "value_name", ...}`. La "exploto" a una tabla larga (una fila por atributo) para contar qué atributos aparecen y qué % de `new` tienen.

In [3]:
print("Publicaciones con al menos un atributo:", round((train["n_atributos"] > 0).mean(), 3))

filas_atributos = [
    (fila, atributo.get("id"), atributo.get("value_name"))
    for fila, lista in train["attributes"].items()
    for atributo in lista
]
atributos = pd.DataFrame(filas_atributos, columns=["fila", "id", "valor"])
atributos["new"] = y.loc[atributos["fila"]].to_numpy()

atributos.groupby("id").agg(n=("new", "size"), pct_new=("new", "mean")) \
    .sort_values("n", ascending=False).head(15).round(3)

Publicaciones con al menos un atributo: 0.124


,n,pct_new
id,,
Season,5399,0.589
GENDER,5256,0.596
MLA-PART_NUMBER,1186,0.967
PART_NUMBER,1161,0.972
MLA1743-HORPREF,789,0.148
MLA1744-MARC,682,0.076
MLA1744-MODL,682,0.076
MLA1744-KMTS,682,0.076
MLA1744-COMBUS,682,0.076


**Lectura.** Solo el 12% de las publicaciones tiene atributos, y casi todos son **propios de una categoría**: `MLA1744-*` son autos (marca, km, puertas, ~8% `new`), `MLA1459-*` inmuebles, `Season`/`GENDER` ropa. Esa información ya la tiene `category_id` (con `TargetEncoder`), así que es redundante.

La excepción es **`PART_NUMBER` / `MLA-PART_NUMBER`** (número de pieza del fabricante): ~1.200 publicaciones con 97% `new`, y aparece en varias categorías. Es el único atributo que vale la pena probar, como una flag.

## EDA 2 — `pictures`

Cada foto trae `max_size` (resolución, ej. `"1200x900"`), `size`, `quality` y un `id`. Primero chequeo los campos que podrían servir.

In [4]:
calidades = pd.Series([foto.get("quality") for lista in train["pictures"] for foto in lista])
print("Valores de quality:", calidades.value_counts().to_dict())

primera_foto = train["pictures"].map(lambda lista: lista[0] if lista else {})
print("Publicaciones con al menos una foto:", round((train["n_fotos"] > 0).mean(), 3))

pct_new_por_grupo(primera_foto.map(lambda foto: foto.get("max_size"))).sort_values("n", ascending=False).head(12)

Valores de quality: {'': 205481}
Publicaciones con al menos una foto: 0.992


,n,pct_del_total,pct_new
pictures,,,
1200x900,10904,0.157,0.348
900x1200,7131,0.103,0.265
640x480,1640,0.024,0.297
1200x675,1472,0.021,0.385
500x500,1259,0.018,0.946
480x640,1125,0.016,0.130
675x1200,1090,0.016,0.368
1200x1200,792,0.011,0.789
800x800,721,0.010,0.979


**Lectura.** `quality` está siempre vacío: no sirve. Pero `max_size` muestra un patrón muy claro, que tiene sentido si se piensa **quién saca la foto**:

- `1200x900` / `900x1200` (proporción 4:3, la de la cámara de un celular o cámara digital): **26–35% `new`**. Un particular le saca una foto al objeto que vende.
- `500x500`, `1000x1000`, `800x800` (**cuadradas**): **79–95% `new`**. Típico de imágenes de catálogo o editadas por un comercio.

Para resumirlo armo, para todas las fotos de cada publicación, el ancho y el alto, y miro tres cosas: resolución, proporción cuadrada o de cámara, y cuántos tamaños distintos hay.

In [5]:
def medidas_fotos(lista_fotos):
    """Ancho y alto (en píxeles) de cada foto, a partir de max_size "ANCHOxALTO"."""
    medidas = []
    for foto in lista_fotos:
        try:
            ancho, alto = foto["max_size"].split("x")
            medidas.append((int(ancho), int(alto)))
        except (KeyError, AttributeError, ValueError):
            continue
    return medidas


def es_proporcion_camara(ancho, alto):
    return abs(ancho / alto - 4 / 3) < 0.01 or abs(ancho / alto - 3 / 4) < 0.01


medidas = train["pictures"].map(medidas_fotos)

train["foto_log_px_primera"] = medidas.map(lambda m: np.log(m[0][0] * m[0][1]) if m else np.nan)
train["foto_pct_cuadradas"] = medidas.map(lambda m: np.mean([a == h for a, h in m]) if m else np.nan)
train["foto_pct_camara"] = medidas.map(lambda m: np.mean([es_proporcion_camara(a, h) for a, h in m]) if m else np.nan)
train["foto_n_tamanios"] = medidas.map(lambda m: len(set(m)))

columnas_fotos = ["foto_log_px_primera", "foto_pct_cuadradas", "foto_pct_camara", "foto_n_tamanios"]

train[columnas_fotos].groupby(y).median().rename(index={0: "used", 1: "new"}).round(3)

,foto_log_px_primera,foto_pct_cuadradas,foto_pct_camara,foto_n_tamanios
condition,,,,
used,13.799,0.0,1.0,1.0
new,13.051,0.0,0.0,2.0


In [6]:
pct_new_por_grupo(pd.qcut(train["foto_log_px_primera"], q=5))

,n,pct_del_total,pct_new
foto_log_px_primera,,,
"(-0.001, 12.137]",13941,0.201,0.770
"(12.137, 13.096]",13829,0.199,0.611
"(13.096, 13.773]",13964,0.201,0.526
"(13.773, 13.892]",23343,0.336,0.356
"(13.892, 14.18]",4347,0.063,0.594


In [7]:
print("Primera foto cuadrada:")
print(pct_new_por_grupo(medidas.map(lambda m: bool(m) and m[0][0] == m[0][1])))
print()
print("Primera foto con proporción de cámara (4:3 o 3:4):")
print(pct_new_por_grupo(medidas.map(lambda m: bool(m) and es_proporcion_camara(*m[0]))))
print()
print("Cantidad de tamaños distintos entre sus fotos:")
print(pct_new_por_grupo(train["foto_n_tamanios"].clip(upper=4)))

Primera foto cuadrada:
              n  pct_del_total  pct_new
pictures                               
False     61469          0.878    0.490
True       8531          0.122    0.876

Primera foto con proporción de cámara (4:3 o 3:4):
              n  pct_del_total  pct_new
pictures                               
False     41234          0.589    0.662
True      28766          0.411    0.358

Cantidad de tamaños distintos entre sus fotos:
                     n  pct_del_total  pct_new
foto_n_tamanios                               
0                  576          0.008    0.316
1                39067          0.558    0.455
2                15485          0.221    0.530
3                 6851          0.098    0.754
4                 8021          0.115    0.783


**Lectura.**

- **Resolución**: cuanto más chica la primera foto, más `new` (de ~77% en el quintil más bajo a ~36% en el de ~1 megapíxel). Fotos de catálogo livianas vs fotos de cámara a resolución completa.
- **Proporción**: las cuadradas son ~88% `new`, las de cámara mucho menos.
- **Tamaños distintos**: las `used` suelen tener todas las fotos del mismo tamaño (la misma cámara). Las `new` mezclan tamaños (imágenes juntadas de distintos lados).

Las cuatro features (`foto_log_px_primera`, `foto_pct_cuadradas`, `foto_pct_camara`, `foto_n_tamanios`) son determinísticas (no aprenden nada de los datos), así que van afuera del `Pipeline`, igual que `aplanar_anidadas`. Las publicaciones sin fotos (0,8%) quedan en `NaN`: `HistGradientBoostingClassifier` maneja nulos de forma nativa (manda los `NaN` a la rama del split que más reduce el error), así que no hace falta imputar.

Descarté el `id` de la foto: trae un sufijo mes-año, pero la diferencia con `start_time` separa poco (entre 51% y 64% `new` en todos los rangos).

## EDA 3 — `variations`

In [8]:
print("Publicaciones con variaciones:", round((train["n_variaciones"] > 0).mean(), 3))
pct_new_por_grupo(train["n_variaciones"].clip(upper=3))

Publicaciones con variaciones: 0.082


,n,pct_del_total,pct_new
n_variaciones,,,
0,64244,0.918,0.534
1,4202,0.060,0.423
2,258,0.004,0.853
3,1296,0.019,0.980


**Lectura.** Solo el 8% de las publicaciones tiene variaciones (talle/color), pero ahí la señal es **fuerte y no lineal**: con 1 variación hay 42% `new`, con 2 sube a 85% y con 3 o más a 98%. Esa señal ya está en la Base, porque `n_variaciones` entra tal cual y un modelo de árboles corta en los valores que hagan falta (no necesita que la relación sea lineal). El resto de cada variación (talle, color, `picture_ids`) se repite con `attributes`/`pictures`. **No armo features nuevas de `variations`.**

## Bloque N — + fotos

Suma las 4 columnas de fotos a la Base (T2a).

In [9]:
evaluar("N - + fotos", armar_pipeline_hgb(columnas_fotos))

N - + fotos -> accuracy: 0.9309 +/- 0.0014 | roc_auc: 0.9814


## Bloque O — + `tiene_part_number`

Flag: la publicación tiene el atributo `PART_NUMBER` o `MLA-PART_NUMBER`. Se suma sobre N.

In [10]:
train["tiene_part_number"] = train["attributes"].map(
    lambda lista: any(atributo.get("id") in ("PART_NUMBER", "MLA-PART_NUMBER") for atributo in lista)
)
print("Con PART_NUMBER:", train["tiene_part_number"].sum(), "| % new:", round(y[train["tiene_part_number"]].mean(), 3))

evaluar("O - + tiene_part_number", armar_pipeline_hgb(columnas_fotos + ["tiene_part_number"]))

Con PART_NUMBER: 1339 | % new: 0.969


O - + tiene_part_number -> accuracy: 0.9311 +/- 0.0016 | roc_auc: 0.9814


## Tabla final y conclusión

In [11]:
tabla_anidadas = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_anidadas["ganancia_accuracy"] = tabla_anidadas["accuracy_media"].diff()
tabla_anidadas.round(4)

,accuracy_media,accuracy_std,roc_auc_media,ganancia_accuracy
bloque,,,,
Base (T2a),0.9291,0.0017,0.9807,NaN
N - + fotos,0.9309,0.0014,0.9814,0.0018
O - + tiene_part_number,0.9311,0.0016,0.9814,0.0002


**Conclusión.**

| Bloque | accuracy | ganancia | roc_auc |
|---|---|---|---|
| Base (T2a) | 0,9291 ± 0,0017 | — | 0,9807 |
| N: + fotos | 0,9309 ± 0,0014 | +0,0018 | 0,9814 |
| O: + `tiene_part_number` | 0,9311 ± 0,0016 | +0,0002 | 0,9814 |

- **EDA.** Lo único nuevo con señal clara son las **fotos**: resolución, proporción cuadrada vs de cámara y cantidad de tamaños distintos separan mucho las clases, y están en el 99% de las publicaciones. `attributes` es casi todo redundante con `category_id`. `variations` ya está cubierta por `n_variaciones`. `quality` está vacío.
- **N (fotos, +0,0018).** Mejora las dos métricas, pero la ganancia es **del orden de un desvío entre folds**. Es exactamente el caso de las fechas en `07` (+0,0018, no se sumaron) y de T3 en `09` (+0,0011). Con el criterio que venimos usando, queda **al límite del ruido**. Era esperable que fuera menos que lo que sugería el EDA: un comercio que sube fotos cuadradas de catálogo ya se distingue por vendedor, tipo de publicación, título, etc.
- **O (`PART_NUMBER`, +0,0002).** Ruido: esas ~1.300 publicaciones (97% `new`) el modelo ya las acertaba. **No se suma.**

**Observación de método (a decidir).** Ya juntamos **cuatro** ganancias marginales, cada una ≈ 1 desvío: T3 (garantía), N (fotos), L (fechas) y M (precio). Por separado ninguna pasa el criterio. Pero todas van en la misma dirección y en las dos métricas, y el desvío entre folds es una vara **exigente**: mide cuánto cambia la accuracy de un fold a otro, no cuánto cambia la **diferencia** entre dos configuraciones medidas **sobre los mismos folds**. Una forma más justa de decidir es comparar fold por fold (¿mejora en los 5 folds o solo en promedio?) y medir las cuatro juntas. Lo propongo para el Paso 5.
